# e2m tutorial: TCGA LUAD

This notebook follows the manuscript workflow for one cancer type. It downloads cohort-level Xena STAR counts and MC3 data, evaluates mutation and TMB prediction with held-out folds, trains a reusable multitask model, and interprets one mutation output with two SHAP methods.

Run the notebook from the repository root after installing `pip install -e ".[interpretation]"`. The model and SHAP cells can take substantial time.

In [ ]:
from pathlib import Path
import pandas as pd

from e2m import cross_validate, download, embed, explain, head_weights, predict_tmb, train

CANCERS = ["LUAD"]
DATA_DIR = Path("e2m_data")
RESULT_DIR = Path("results/luad")
MODEL_DIR = Path("models/luad")
DATA_OPTIONS = {"expression_dataset": "star_counts", "expression_transform": "raw"}
RESULT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Download and prepare data

The manuscript setup keeps GENCODE v36 protein-coding genes, averages duplicate symbols, retains mutation targets present in at least 5 percent of matched samples, and caps the panel at 400 targets.

In [ ]:
download_summary = download(
    CANCERS,
    data_dir=DATA_DIR,
    data_overrides=DATA_OPTIONS,
    output_dir=RESULT_DIR / "data",
)
download_summary

In [ ]:
expression = pd.read_csv(RESULT_DIR / "data/expression.csv.gz", index_col=0)
mutations = pd.read_csv(RESULT_DIR / "data/mutations.csv.gz", index_col=0)
expression.shape, mutations.shape, mutations.mean().sort_values(ascending=False).head(10)

## 2. Held-out mutation prediction

Use these out-of-fold results for performance reporting. Normalized AUPRC is 0 at the prevalence baseline and 1 for perfect ranking.

In [ ]:
mutation_cv = cross_validate(
    CANCERS,
    output_dir=RESULT_DIR / "mutation_cv",
    data_dir=DATA_DIR,
    data_overrides=DATA_OPTIONS,
)
mutation_cv

In [ ]:
mutation_metrics = pd.read_csv(RESULT_DIR / "mutation_cv/metrics.csv", index_col=0)
mutation_metrics.sort_values("normalized_auprc", ascending=False).head(15)

## 3. Held-out TMB prediction

Coding MC3 events define TMB. Samples without a coding mutation event are dropped. XGBoost predicts `log2(TMB + 1)`.

In [ ]:
tmb_cv = predict_tmb(
    CANCERS,
    output_dir=RESULT_DIR / "tmb_cv",
    data_dir=DATA_DIR,
    data_overrides=DATA_OPTIONS,
)
pd.read_csv(RESULT_DIR / "tmb_cv/summary.csv")

## 4. Train a reusable multitask model

This full-cohort fit is for deployment, embeddings, head weights, and direct neural SHAP. It is not a held-out performance estimate.

In [ ]:
model_summary = train(
    CANCERS,
    output_dir=MODEL_DIR,
    data_dir=DATA_DIR,
    data_overrides=DATA_OPTIONS,
)
model_summary

## 5. Explain TP53 with manuscript Tree SHAP

This separate full-cohort XGBoost classifier is used for attribution only. It does not replace the multitask model used for mutation performance.

In [ ]:
xgb_shap = explain(
    MODEL_DIR,
    target="TP53",
    method="xgboost",
    output_dir=RESULT_DIR / "shap_xgboost_tp53",
    data_dir=DATA_DIR,
)
pd.read_csv(RESULT_DIR / "shap_xgboost_tp53/feature_summary.csv").head(20)

## 6. Explain the multitask TP53 output directly

Gradient SHAP explains the neural probability output using a sampled LUAD background. This is useful for inspecting the trained network, but the manuscript interpretation uses the XGBoost method above.

In [ ]:
neural_shap = explain(
    MODEL_DIR,
    target="TP53",
    method="neural",
    output_dir=RESULT_DIR / "shap_neural_tp53",
    data_dir=DATA_DIR,
)
pd.read_csv(RESULT_DIR / "shap_neural_tp53/feature_summary.csv").head(20)

## 7. Export embeddings and output-head weights

In [ ]:
embedding_summary = embed(
    MODEL_DIR,
    RESULT_DIR / "data/expression.csv.gz",
    RESULT_DIR / "sample_embeddings.csv",
)
weight_summary = head_weights(MODEL_DIR, RESULT_DIR / "head_weights.csv")
embedding_summary, weight_summary

## Interpretation note

SHAP reports features used by a model. In bulk RNA data, those features can reflect mutation-associated expression, subtype, co-mutation, immune cells, stromal cells, or other correlated biology. Do not treat a SHAP association as proof of a direct causal effect.